In [1]:
import pandas as pd
import numpy as np
import duckdb
import plotly.express as px
import plotly.graph_objects as go

### Import files


#### Budget des communes


In [2]:
# Optimisation du chargement des données en limitant les types de données
types_optimises = {
    "Exercice": "int32",
    "Code Insee 2024 Commune": "str",
    "Nom 2024 Commune": "str",
    "Population totale": "float64",
    "Type de budget": "str",
    "Nomenclature": "str",
    "Agrégat": "str",
    "Montant": "float64",
}

# Chargement des données en ne sélectionnant que certaines colonnes
budget_original = pd.read_csv(
    "C:/Users/lesli/kDrive2/Data/Data For Good/14_prixchangementclimatique/Sources/ofgl-base-communes.csv",
    sep=";",
    usecols=[
        "Exercice",
        "Code Insee 2024 Commune",
        "Nom 2024 Commune",
        "Population totale",
        "Type de budget",
        "Nomenclature",
        "Agrégat",
        "Montant",
    ],
    dtype=types_optimises,
    on_bad_lines="skip",
)

# DataFrame
budget_original

,Exercice,Code Insee 2024 Commune,Nom 2024 Commune,Type de budget,Nomenclature,Agrégat,Montant,Population totale
0,2017,69175,Savigny,Budget principal,M14,Epargne brute,319673.98,2073.0
1,2017,69175,Savigny,Budget annexe,M14,Epargne brute,16791.97,2073.0
2,2017,69176,Soucieu-en-Jarrest,Budget principal,M14,Epargne brute,517111.78,4331.0
3,2017,69177,Sourcieux-les-Mines,Budget principal,M14,Epargne brute,268090.53,2060.0
4,2017,69178,Souzy,Budget principal,M14,Epargne brute,135733.30,805.0
...,...,...,...,...,...,...,...,...
22364882,2024,29277,Sizun,Budget annexe,M49A,Variation du fonds de roulement,0.00,2398.0
22364883,2024,36006,Argenton-sur-Creuse,Budget annexe,M57,Variation du fonds de roulement,0.00,5072.0
22364884,2024,44186,Sainte-Pazanne,Budget annexe,M57,Variation du fonds de roulement,0.00,7357.0
22364885,2024,69059,Civrieux-d'Azergues,Budget annexe,M57,Variation du fonds de roulement,0.00,1625.0


In [ ]:
# Copie du dataset
budget = budget_original

#### Comptes des communes


In [4]:
comptes_original = pd.read_csv(
    "C:/Users/lesli/kDrive2/Data/Data For Good/14_prixchangementclimatique/Sources/Balance_Commune_2024_Dec2025.csv",
    sep=";",
    usecols=[
        "IDENT",
        "LBUDG",
        "INSEE",
        "CTYPE",
        "NOMEN",
        "BAL",
        "COMPTE",
        #    'BEDEB',
        #    'BECRE',
        #    'OBNETDEB',
        #    'OBNETCRE',
        #    'ONBDEB',
        #    'ONBCRE',
        #    'OOBDEB',
        #    'OOBCRE',
        "SD",
        #    'SC'
    ],
    encoding="latin-1",
    on_bad_lines="skip",
)

In [ ]:
# Copie du dataset
comptes = comptes_original
comptes

,IDENT,LBUDG,INSEE,CTYPE,NOMEN,BAL,COMPTE,SD
0,20005339500014,LE POIZAT-LALLEYRIAT,204.0,101,M57A,DEF,1021,"0,00"
1,20005339500014,LE POIZAT-LALLEYRIAT,204.0,101,M57A,DEF,10222,"0,00"
2,20005339500014,LE POIZAT-LALLEYRIAT,204.0,101,M57A,DEF,10226,"0,00"
3,20005339500014,LE POIZAT-LALLEYRIAT,204.0,101,M57A,DEF,10227,"0,00"
4,20005339500014,LE POIZAT-LALLEYRIAT,204.0,101,M57A,DEF,10228,"0,00"
...,...,...,...,...,...,...,...,...
7036730,20000888600018,TSINGONI,617.0,101,M57,DEF,7518,"0,00"
7036731,20000888600018,TSINGONI,617.0,101,M57,DEF,752,"0,00"
7036732,20000888600018,TSINGONI,617.0,101,M57,DEF,75888,"0,00"
7036733,20000888600018,TSINGONI,617.0,101,M57,DEF,7688,"0,00"


In [77]:
comptes.describe()

,IDENT,INSEE,CTYPE,COMPTE
count,7.036735e+06,6.129857e+06,7.036735e+06,7.036735e+06
mean,2.141562e+13,2.435348e+02,1.409753e+02,2.101461e+05
std,3.733839e+11,1.779781e+02,1.039947e+02,2.104967e+06
min,2.000022e+13,1.000000e+00,1.010000e+02,1.200000e+01
25%,2.123047e+13,1.010000e+02,1.010000e+02,2.033000e+03
50%,2.145028e+13,2.100000e+02,1.010000e+02,6.411000e+03
75%,2.167044e+13,3.490000e+02,1.010000e+02,6.061100e+04
max,2.197402e+13,9.090000e+02,7.020000e+02,6.573622e+07


#### Communes non assurées


In [6]:
non_assurees = pd.read_csv(
    "C:/Users/lesli/kDrive2/Data/Data For Good/14_prixchangementclimatique/Sources/communes_non_assurees.csv"
)
non_assurees

,EXER,CODGEO,LBUDG,is_insured
0,2017.0,01066,BURBANCHE (LA ),False
1,2017.0,01311,COMMERCES SAVIGNEUX,False
2,2017.0,01311,PREMILLIEU,False
3,2017.0,02232,COYOLLES,False
4,2017.0,02478,MERLIEUX-ET-FOUQUEROLLES,False
...,...,...,...,...
1511,2024.0,91148,CHAUFFOUR-LES-ETRECHY,False
1512,2024.0,91393,MEROBERT,False
1513,2024.0,97401,LES AVIRONS,False
1514,2024.0,97414,SAINT-LOUIS,False


#### Référentiel INSEE


In [ ]:
# Connexion à la base de données
con = duckdb.connect("odis.duckdb")

# Récupération de la table
table_name = "gold_gold_com_dep_reg"
com_dep_reg = con.sql(f"SELECT * FROM {table_name}")

ref_insee = pd.DataFrame(data=com_dep_reg)
ref_insee

ValueError: DataFrame constructor not properly called!

In [63]:
com_dep_reg

┌─────────┬─────────┬─────────┬────────────────────────────┐
│ CODGEO  │ CODREG  │ CODDEP  │           LIBGEO           │
│ varchar │ varchar │ varchar │          varchar           │
├─────────┼─────────┼─────────┼────────────────────────────┤
│ 01001   │ 84      │ 01      │ L'Abergement-Clémenciat    │
│ 01002   │ 84      │ 01      │ L'Abergement-de-Varey      │
│ 01004   │ 84      │ 01      │ Ambérieu-en-Bugey          │
│ 01005   │ 84      │ 01      │ Ambérieux-en-Dombes        │
│ 01006   │ 84      │ 01      │ Ambléon                    │
│ 01007   │ 84      │ 01      │ Ambronay                   │
│ 01008   │ 84      │ 01      │ Ambutrix                   │
│ 01009   │ 84      │ 01      │ Andert-et-Condon           │
│ 01010   │ 84      │ 01      │ Anglefort                  │
│ 01011   │ 84      │ 01      │ Apremont                   │
│   ·     │ ·       │ ·       │    ·                       │
│   ·     │ ·       │ ·       │    ·                       │
│   ·     │ ·       │ · 

### Cleaning


#### Renommage des colonnes


In [6]:
# Budgets
budget = budget.rename(
    columns={
        "Exercice": "exercice",
        "Code Insee 2024 Commune": "code_insee_commune",
        "Nom 2024 Commune": "nom_commune",
        "Type de budget": "budget_type",
        "Nomenclature": "nomenclature",
        "Agrégat": "agregat",
        "Montant": "montant",
        "Population totale": "pop_totale",
    }
)

#### Filtres


In [ ]:
# Filtre sur l'année 2024
budget_24 = budget.loc[budget["exercice"] == 2024].copy()

In [7]:
# Filtres sur les budgets des communes
primes_communes = comptes.loc[
    (
        (comptes["CTYPE"] == 410)  # communes budget principal
        | (comptes["CTYPE"] == 101)  # communes budget annexe
    )
    & (comptes["COMPTE"] == 616)  # code du montant des primes d'assurance
]
primes_communes

,IDENT,LBUDG,INSEE,CTYPE,NOMEN,BAL,COMPTE,SD
2643,20007722000073,ASSAINISSEMENT BAGE-DOMMARTIN,NaN,410,M49A,DEF,616,"3500,41"
5581,21010001200033,ASST ABERGEMENT CLEMENCIAT,NaN,410,M49A,DEF,616,"1082,95"
34291,21010159800055,ASST FEILLENS,NaN,410,M49A,DEF,616,"12228,53"
40284,21010199400049,EAU ASST JUJURIEUX,NaN,410,M49A,DEF,616,"2885,66"
44593,21010231500046,ASST MANZIAT,NaN,410,M49A,DEF,616,"1869,47"
...,...,...,...,...,...,...,...,...
6835624,21890449800068,EAU VILLEBLEVIN,NaN,410,M49A,DEF,616,"3485,02"
6839230,21890467000054,ASST VILLETHIERRY,NaN,410,M49A,DEF,616,"1696,76"
6839770,21890469600034,EAU ASST PERCENEIGE,NaN,410,M49A,DEF,616,"1168,65"
6840442,21890472000040,ASST VILLIERS SAINT BENOIT,NaN,410,M49A,DEF,616,"392,98"


#### Types conversion


In [9]:
# Conversion de l'identification en string
primes_communes["IDENT"] = primes_communes["IDENT"].astype("str")

C:\Users\lesli\AppData\Local\Temp\ipykernel_9812\80754677.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  primes_communes["IDENT"] = primes_communes["IDENT"].astype("str")


In [10]:
# Conversion des colonnes texte en numéric en remplacant la virgule par le point
primes_communes["SD"] = primes_communes["SD"].str.replace(",", ".").astype("float64")

C:\Users\lesli\AppData\Local\Temp\ipykernel_9812\2654778666.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  primes_communes["SD"] = primes_communes["SD"].str.replace(",", ".").astype("float64")


#### Création de la colonne INSEE à partir de l'identification


In [ ]:
# Code INSEE = Caractères 2 & 3 + 4 à 7
primes_communes["insee"] = (
    primes_communes["IDENT"].str[2:4] + primes_communes["IDENT"].str[5:8]
)
primes_communes

C:\Users\lesli\AppData\Local\Temp\ipykernel_9812\659017200.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  primes_communes["insee"] = (


,IDENT,LBUDG,INSEE,CTYPE,NOMEN,BAL,COMPTE,SD,insee
2643,20007722000073,ASSAINISSEMENT BAGE-DOMMARTIN,NaN,410,M49A,DEF,616,3500.41,00722
5581,21010001200033,ASST ABERGEMENT CLEMENCIAT,NaN,410,M49A,DEF,616,1082.95,01001
34291,21010159800055,ASST FEILLENS,NaN,410,M49A,DEF,616,12228.53,01159
40284,21010199400049,EAU ASST JUJURIEUX,NaN,410,M49A,DEF,616,2885.66,01199
44593,21010231500046,ASST MANZIAT,NaN,410,M49A,DEF,616,1869.47,01231
...,...,...,...,...,...,...,...,...,...
6835624,21890449800068,EAU VILLEBLEVIN,NaN,410,M49A,DEF,616,3485.02,89449
6839230,21890467000054,ASST VILLETHIERRY,NaN,410,M49A,DEF,616,1696.76,89467
6839770,21890469600034,EAU ASST PERCENEIGE,NaN,410,M49A,DEF,616,1168.65,89469
6840442,21890472000040,ASST VILLIERS SAINT BENOIT,NaN,410,M49A,DEF,616,392.98,89472


#### Conversion du code commune INSEE en 5 caractère (si nécessaire)


In [12]:
# Vérification du nombre de caractère du code Insee
budget_24["code_insee_len"] = budget_24["code_insee_commune"].str.len()
budget_24.groupby(["code_insee_len"])["code_insee_commune"].nunique()
# budget_24.loc[budget_24['code_insee_len'] == 1]

code_insee_len
5    34932
Name: code_insee_commune, dtype: int64

In [ ]:
# budget_24['code_insee_clean'] = (pd.to_numeric(budget_24['code_insee_commune'], errors='coerce').astype(int))
# budget_24["code_insee_clean"] = budget_24["code_insee_clean"].fillna(0).astype(str)
# budget_24.dtypes

exercice                         int64
revenu_par_habitant_tranche    float64
code_insee_commune              object
nom_commune                     object
budget_type                     object
nomenclature                    object
agregat                         object
montant                        float64
pop_totale                     float64
compte_2024                    float64
code_insee_len                   int64
code_insee_clean                object
dtype: object

### Exploration


#### Filtre sur le budget total


In [13]:
# Filtre sur les lines de recettes totale (budget)
budget_24_tot = budget_24.loc[(budget_24["agregat"] == "Dépenses totales")].copy()
budget_24_tot

,exercice,code_insee_commune,nom_commune,budget_type,nomenclature,agregat,montant,pop_totale,code_insee_len
19561418,2024,28050,Boncourt,Budget annexe,M57A,Dépenses totales,50976.31,277.0,5
19561419,2024,28051,Bonneval,Budget principal,M57,Dépenses totales,5257518.82,4955.0,5
19561420,2024,28051,Bonneval,Budget annexe,M49A,Dépenses totales,878496.72,4955.0,5
19561421,2024,28051,Bonneval,Budget annexe,M4,Dépenses totales,150465.47,4955.0,5
19561422,2024,28051,Bonneval,Budget annexe,M57,Dépenses totales,171209.14,4955.0,5
...,...,...,...,...,...,...,...,...,...
19781413,2024,28047,Boisville-la-Saint-Père,Budget annexe,M57A,Dépenses totales,2613.27,732.0,5
19781414,2024,28047,Boisville-la-Saint-Père,Budget annexe,M57A,Dépenses totales,202329.40,732.0,5
19781415,2024,28048,La Bourdinière-Saint-Loup,Budget principal,M57A,Dépenses totales,422509.08,759.0,5
19781416,2024,28049,Boncé,Budget principal,M57A,Dépenses totales,129767.47,241.0,5


In [14]:
budget_24_tot = (
    budget_24_tot.groupby(
        ["code_insee_commune", "nom_commune", "agregat", "pop_totale"]
    )["montant"]
    .sum()
    .reset_index()
)
budget_24_tot

,code_insee_commune,nom_commune,agregat,pop_totale,montant
0,01001,L'Abergement-Clémenciat,Dépenses totales,848.0,1301311.86
1,01002,L'Abergement-de-Varey,Dépenses totales,273.0,675902.79
2,01004,Ambérieu-en-Bugey,Dépenses totales,15240.0,19874057.05
3,01005,Ambérieux-en-Dombes,Dépenses totales,1921.0,1681162.26
4,01006,Ambléon,Dépenses totales,113.0,113384.06
...,...,...,...,...,...
34927,97613,M'Tsangamouji,Dépenses totales,6586.0,14849198.20
34928,97614,Ouangani,Dépenses totales,10393.0,18403039.69
34929,97615,Pamandzi,Dépenses totales,11802.0,11662994.82
34930,97616,Sada,Dépenses totales,11619.0,18530545.20


#### Jointure fichier budget et fichier primes


In [15]:
budget_prime = pd.merge(
    budget_24_tot,
    primes_communes[["insee", "SD"]],
    left_on=["code_insee_commune"],
    right_on=["insee"],
    how="left",
)
budget_prime

,code_insee_commune,nom_commune,agregat,pop_totale,montant,insee,SD
0,01001,L'Abergement-Clémenciat,Dépenses totales,848.0,1301311.86,01001,1082.95
1,01002,L'Abergement-de-Varey,Dépenses totales,273.0,675902.79,NaN,NaN
2,01004,Ambérieu-en-Bugey,Dépenses totales,15240.0,19874057.05,NaN,NaN
3,01005,Ambérieux-en-Dombes,Dépenses totales,1921.0,1681162.26,NaN,NaN
4,01006,Ambléon,Dépenses totales,113.0,113384.06,NaN,NaN
...,...,...,...,...,...,...,...
34982,97613,M'Tsangamouji,Dépenses totales,6586.0,14849198.20,NaN,NaN
34983,97614,Ouangani,Dépenses totales,10393.0,18403039.69,NaN,NaN
34984,97615,Pamandzi,Dépenses totales,11802.0,11662994.82,NaN,NaN
34985,97616,Sada,Dépenses totales,11619.0,18530545.20,NaN,NaN


In [17]:
# Renomme la colonne SD en prime
budget_prime.rename(columns={"SD": "prime"}, inplace=True)

#### Calcul de la part de la prime dans le budget total


In [18]:
# Remplacement des valeurs NaN en 0
budget_prime["prime"] = np.nan_to_num(budget_prime["prime"])

In [ ]:
# Vérification du nombre de lignes sans prime
budget_prime.loc[budget_prime["prime"] == 0]

,code_insee_commune,nom_commune,agregat,pop_totale,montant,insee,prime
1,01002,L'Abergement-de-Varey,Dépenses totales,273.0,675902.79,NaN,0.0
2,01004,Ambérieu-en-Bugey,Dépenses totales,15240.0,19874057.05,NaN,0.0
3,01005,Ambérieux-en-Dombes,Dépenses totales,1921.0,1681162.26,NaN,0.0
4,01006,Ambléon,Dépenses totales,113.0,113384.06,NaN,0.0
5,01007,Ambronay,Dépenses totales,2951.0,4616489.44,NaN,0.0
...,...,...,...,...,...,...,...
34982,97613,M'Tsangamouji,Dépenses totales,6586.0,14849198.20,NaN,0.0
34983,97614,Ouangani,Dépenses totales,10393.0,18403039.69,NaN,0.0
34984,97615,Pamandzi,Dépenses totales,11802.0,11662994.82,NaN,0.0
34985,97616,Sada,Dépenses totales,11619.0,18530545.20,NaN,0.0


In [ ]:
# Calcul de la part de la prime dans le budget total
budget_prime["part_prime"] = budget_prime["prime"] / budget_prime["montant"]
budget_prime

,code_insee_commune,nom_commune,agregat,pop_totale,montant,insee,prime,part_prime
0,01001,L'Abergement-Clémenciat,Dépenses totales,848.0,1301311.86,01001,1082.95,0.000832
1,01002,L'Abergement-de-Varey,Dépenses totales,273.0,675902.79,NaN,0.00,0.000000
2,01004,Ambérieu-en-Bugey,Dépenses totales,15240.0,19874057.05,NaN,0.00,0.000000
3,01005,Ambérieux-en-Dombes,Dépenses totales,1921.0,1681162.26,NaN,0.00,0.000000
4,01006,Ambléon,Dépenses totales,113.0,113384.06,NaN,0.00,0.000000
...,...,...,...,...,...,...,...,...
34982,97613,M'Tsangamouji,Dépenses totales,6586.0,14849198.20,NaN,0.00,0.000000
34983,97614,Ouangani,Dépenses totales,10393.0,18403039.69,NaN,0.00,0.000000
34984,97615,Pamandzi,Dépenses totales,11802.0,11662994.82,NaN,0.00,0.000000
34985,97616,Sada,Dépenses totales,11619.0,18530545.20,NaN,0.00,0.000000


#### Groupement des montants


##### Part de la prime dans le budget


In [89]:
# des catégories
conditions = [
    (budget_prime["part_prime"] == 0),
    (budget_prime["part_prime"] <= 0.001),
    (budget_prime["part_prime"] <= 0.002),
    (budget_prime["part_prime"] <= 0.005),
    (budget_prime["part_prime"] > 0.005),
]

# Catégories
results = ["0. Pas de prime", "1. 0-1%", "2. 1-2%", "3. 2-5%", "4. >5%"]

budget_prime["prime_cat"] = np.select(conditions, results)

In [90]:
budget_prime.groupby(["prime_cat"]).count()

,code_insee_commune,nom_commune,agregat,pop_totale,montant,insee,prime,part_prime
prime_cat,,,,,,,,
0. Pas de prime,34356,34356,34356,34356,34356,1,34356,34356
1. 0-1%,290,290,290,290,290,290,290,290
2. 1-2%,141,141,141,141,141,141,141,141
3. 2-5%,137,137,137,137,137,137,137,137
4. >5%,63,63,63,63,63,63,63,63


##### Tranches de montant du budget


In [ ]:
# des catégories
conditions_budget = [
    (budget_prime["montant"] < 50000),
    (budget_prime["montant"] < 100000),
    (budget_prime["montant"] < 200000),
    (budget_prime["montant"] < 500000),
    (budget_prime["montant"] < 1000000),
    (budget_prime["montant"] < 5000000),
    (budget_prime["montant"] >= 5000000),
]

# Catégories
results_budget = [
    "1. 0 - 50K",
    "2. 50K-100K",
    "3. 100K-200K",
    "4. 200K-500K",
    "5. 500K-1M",
    "6. 1M -5M",
    "7.>5M",
]

budget_prime["montant_cat"] = np.select(conditions_budget, results_budget)

In [103]:
budget_prime.groupby(["montant_cat"]).count()

,code_insee_commune,nom_commune,agregat,pop_totale,montant,insee,prime,part_prime,prime_cat
montant_cat,,,,,,,,,
1. 0 - 50K,432,432,432,432,432,2,432,432,432
2. 50K-100K,2130,2130,2130,2130,2130,14,2130,2130,2130
3. 100K-200K,5099,5099,5099,5099,5099,51,5099,5099,5099
4. 200K-500K,9149,9149,9149,9149,9149,132,9149,9149,9149
5. 500K-1M,6335,6335,6335,6335,6335,162,6335,6335,6335
6. 1M -5M,8389,8389,8389,8389,8389,218,8389,8389,8389
7.>5M,3453,3453,3453,3453,3453,53,3453,3453,3453


### Visualisation


In [91]:
fig_pie = px.pie(budget_prime, names="prime_cat")
fig_pie.show()

In [ ]:
fig_pie2 = px.pie(
    budget_prime.loc[budget_prime["pop_totale"] > 3500], names="prime_cat"
)
fig_pie2.show()

In [93]:
# Catégories par ordre alphabétique
sort = sorted(budget_prime["prime_cat"].unique())

# Histogramme excluant les communes sans prime d'assurance
fig_histo = px.histogram(
    budget_prime.loc[budget_prime["prime_cat"] != "0. Pas de prime"],
    x="prime_cat",
    category_orders={"prime_cat": sort},
)
fig_histo.show()

In [ ]:
fig_box = px.scatter(
    budget_prime.loc[budget_prime["prime_cat"] != "0. Pas de prime"],
    x="prime_cat",
    y="montant",
    category_orders={"prime_cat": sort},
)
fig_box.show()

In [ ]:
# Catégories par ordre alphabétique
sort2 = sorted(budget_prime["montant_cat"].unique())

# Histogramme excluant les communes sans prime d'assurance
fig_bar = px.histogram(
    budget_prime.loc[budget_prime["prime"] != 0],
    x="montant_cat",
    color="prime_cat",
    category_orders={"montant_cat": sort2, "prime_cat": sort},
    barnorm="percent",
)

fig_bar.update_layout(
    title="Répartion de la part de la prime dans les budget des communes 2024 (communes sans primes exclues)",
    yaxis_title="Pourcentage (%)",
    xaxis_title="Montant du budget",
    legend_title="Part de la prime",
)

fig_bar.show()

### Old - no longer used


#### Vérifications des colonnes nécessaires


##### Colonnes de population totale


In [ ]:
# Vérifier les différences entre population totale et population totale du dernier exercice
budget_24["pop_check"] = np.where(
    budget_24["pop_totale"] == budget_24["pop_totale_dernier_exercice"], True, False
)

C:\Users\lesli\AppData\Local\Temp\ipykernel_4908\3050632631.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  budget_24['pop_check'] = np.where(budget_24['pop_totale'] == budget_24['pop_totale_dernier_exercice'], True, False)


In [ ]:
budget_24.groupby(["pop_check"]).count()

,exercice,revenu_par_habitant_tranche,code_insee_commune,nom_commune,code_insee_collectivite,budget_type,nomenclature,agregat,montant,pop_totale,compte_2024,pop_totale_dernier_exercice
pop_check,,,,,,,,,,,,
True,2660881,2660881,2660881,2660881,2660881,2660881,2660881,2660881,2660881,2660881,2660881,2660881


In [ ]:
budget_24.loc[budget_24["pop_check"] == False]

,exercice,revenu_par_habitant_tranche,code_insee_commune,nom_commune,code_insee_collectivite,budget_type,nomenclature,agregat,montant,pop_totale,compte_2024,pop_totale_dernier_exercice,pop_check


##### Colonnes code INSEE (commune & collectivité)


In [ ]:
budget_24["code_insee_check"] = np.where(
    budget_24["code_insee_commune"] == budget_24["code_insee_collectivite"], True, False
)
budget_24.groupby(["code_insee_check"]).count()

C:\Users\lesli\AppData\Local\Temp\ipykernel_4908\130349152.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  budget_24['code_insee_check'] = np.where(budget_24['code_insee_commune'] == budget_24['code_insee_collectivite'], True, False)


,exercice,revenu_par_habitant_tranche,code_insee_commune,nom_commune,code_insee_collectivite,budget_type,nomenclature,agregat,montant,pop_totale,compte_2024,pop_totale_dernier_exercice,pop_check
code_insee_check,,,,,,,,,,,,,
True,2660881,2660881,2660881,2660881,2660881,2660881,2660881,2660881,2660881,2660881,2660881,2660881,2660881


#### Test


In [ ]:
test_commune = budget_24_tot.loc[
    (budget_24_tot)["code_insee_clean"] == "01066"
].sort_values(by=["nomenclature"])
test_commune
# test_commune.groupby(['code_insee_commune','agregat'])['montant'].sum()

,exercice,revenu_par_habitant_tranche,code_insee_commune,nom_commune,budget_type,nomenclature,agregat,montant,pop_totale,compte_2024,code_insee_len,code_insee_clean
19771784,2024,1.0,1066,La Burbanche,Budget principal,M57A,Dépenses totales,137718.69,96.0,1.0,5,01066
22289976,2024,1.0,1066,La Burbanche,Budget principal,M57A,Recettes totales,168445.33,96.0,1.0,5,01066


In [ ]:
# Convertir en numérique, remplir les vides par 0, transformer en entier, puis en texte
budget_24["code_insee_clean"] = (
    pd.to_numeric(budget_24["code_insee_commune"], errors="coerce")
    .fillna(0)
    .astype(int)
    .astype(str)
    .str.zfill(5)  # Ajoute le 0 au début si besoin pour faire 5 caractères
)